# ETL Silver - Precipitacion diaria Salto Grande

Publica lluvia diaria acumulada por estacion SG en una tabla Silver separada.

In [ ]:
from datetime import timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F

BRONZE_TABLE = 'weather.bronze.sg_rainfall'
TARGET_TABLE = 'weather.silver.sg_rainfall_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
THRESHOLD_PCT = 0.90

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '30')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 30

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def apply_incremental_window(df):
    if load_mode == 'full':
        return df

    max_target_fecha = spark.table(TARGET_TABLE).agg(F.max('fecha').alias('max_fecha')).first()['max_fecha']
    if max_target_fecha is None:
        return df

    start_date = max_target_fecha - timedelta(days=incremental_lookback_days)
    print(f'Processing SG rainfall from {start_date}')
    return df.filter(F.col('fecha') >= F.lit(start_date))


def build_daily(df):
    return (
        df.filter(F.col('fecha').isNotNull())
        .filter(F.col('id_estacion').isNotNull())
        .filter((F.col('p').isNull()) | (F.col('p') >= F.lit(0.0)))
        .groupBy('fecha', 'id_estacion')
        .agg(
            F.first('nombre', ignorenulls=True).alias('nombre'),
            F.first('latitud', ignorenulls=True).alias('latitud'),
            F.first('longitud', ignorenulls=True).alias('longitud'),
            F.sum('p').alias('lluvia_acumulada_mm'),
            F.count('*').cast('bigint').alias('registros_total'),
            F.count('p').cast('bigint').alias('registros_validos'),
        )
        .withColumn('source_table', F.lit(BRONZE_TABLE))
        .withColumn('processed_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select(
            'fecha', 'id_estacion', 'nombre', 'latitud', 'longitud', 'lluvia_acumulada_mm',
            'registros_total', 'registros_validos', 'source_table', 'processed_at', 'updated_at'
        )
    )


def build_quality(df):
    return (
        df.agg(
            F.min('fecha').alias('evaluation_start_date'),
            F.max('fecha').alias('evaluation_end_date'),
            F.countDistinct(F.when(F.col('lluvia_acumulada_mm').isNotNull(), F.col('fecha'))).cast('bigint').alias('observed_days'),
        )
        .withColumn('expected_days', F.when(F.col('evaluation_start_date').isNull(), F.lit(0)).otherwise(F.datediff(F.col('evaluation_end_date'), F.col('evaluation_start_date')) + F.lit(1)).cast('bigint'))
        .withColumn('missing_days', F.greatest(F.col('expected_days') - F.col('observed_days'), F.lit(0)).cast('bigint'))
        .withColumn('missing_pct', F.when(F.col('expected_days') == 0, F.lit(1.0)).otherwise(F.col('missing_days') / F.col('expected_days')))
        .withColumn('threshold_pct', F.lit(THRESHOLD_PCT))
        .withColumn('is_usable', F.col('missing_pct') <= F.col('threshold_pct'))
        .withColumn('source_layer', F.lit('silver'))
        .withColumn('source_table', F.lit(TARGET_TABLE))
        .withColumn('source_name', F.lit('sg_rainfall_daily'))
        .withColumn('attribute_name', F.lit('lluvia_acumulada_mm'))
        .withColumn('grain', F.lit('global_source_daily'))
        .withColumn('evaluated_at', F.current_timestamp())
        .withColumn('notes', F.lit('Lluvia Salto Grande diaria acumulada por estaciones con variable P'))
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('source_layer', 'source_table', 'source_name', 'attribute_name', 'grain', 'evaluation_start_date', 'evaluation_end_date', 'expected_days', 'observed_days', 'missing_days', 'missing_pct', 'threshold_pct', 'is_usable', 'evaluated_at', 'notes', 'created_at', 'updated_at')
    )


def merge_quality(quality_df):
    DeltaTable.forName(spark, QUALITY_TABLE).alias('t').merge(
        quality_df.alias('s'),
        't.source_table = s.source_table AND t.attribute_name = s.attribute_name AND t.grain = s.grain',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def merge_daily(daily_df):
    if daily_df.limit(1).count() == 0:
        print('No SG rainfall rows to merge')
        return

    DeltaTable.forName(spark, TARGET_TABLE).alias('t').merge(
        daily_df.alias('s'),
        't.fecha = s.fecha AND t.id_estacion = s.id_estacion',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
bronze_all = spark.table(BRONZE_TABLE)
daily_to_merge = build_daily(apply_incremental_window(bronze_all))
merge_daily(daily_to_merge)

quality_df = build_quality(spark.table(TARGET_TABLE))
merge_quality(quality_df)

quality_df.show(truncate=False)
spark.table(TARGET_TABLE).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows')).show()